In [47]:
import easyocr
from rapidfuzz import fuzz
import cv2
import re
from sklearn.cluster import AgglomerativeClustering
import numpy as np
import re
reader = easyocr.Reader(['en']) # this needs to run only once to load the model into memory


result = reader.readtext('pariscafe2.jpg')

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\Users\enara\anaconda3\envs\ocr\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:

#result = reader.readtext('samplemenu.jpg') #raw ocr read
#print(result)

def to_rect(box):
    xlist = [p[0] for p in box] #0 = x coordinates
    ylist = [p[1] for p in box] #1 = y coordinates
    return {
        "left": min(xlist),
        "right": max(xlist),
        "top": min(ylist),
        "bottom": max(ylist),
        "height": max(ylist) - min(ylist),
        "width": max(xlist) - min(xlist),
        "center_y": (min(ylist) + max(ylist)) / 2,
        "center_x": (min(xlist) + max(xlist)) / 2
    }

def same_line(rect_a, rect_b):
    avg_h = (rect_a["height"] + rect_b["height"]) / 2
    return abs(rect_a["center_y"] - rect_b["center_y"]) < avg_h * 0.4

def structure_data(result):
    structured = []
    for bbox, text, conf in result:
        structured.append({
            "rect": to_rect(bbox),
            "text": text,
            "conf": conf
        })

    return structured    


PRICE_PATTERN = re.compile(r"""
    ^\s*
    [£$€EeLlIi{YS]?        # optional currency-like symbol
    \s*
    \d+                 # digits
    (\.\d{1,2})?        # optional decimal
    \s*$
""", re.VERBOSE)


#k-means clustering for columns: only works for known number of columns
#hierarchical clustering alternative
#alternative: rule based

def compute_line_gaps(lines):
    gaps = []
    for i in range(1, len(lines)):
        prev_bottom = max(b["rect"]["bottom"] for b in lines[i-1])
        curr_top = min(b["rect"]["top"] for b in lines[i])
        gaps.append(curr_top - prev_bottom)
    return gaps

def find_gap_threshold(gaps):
    positive = [g for g in gaps if g > 0]
    if not positive:
        return 9999
    median_gap = np.median(positive)
    return median_gap * 1.2

def group_lines_into_items(lines):
    gaps = compute_line_gaps(lines)
    threshold = find_gap_threshold(gaps)

    items = []
    current_item = []

    for i, line in enumerate(lines):
        if i == 0:
            current_item.append(line)
            continue

        if gaps[i-1] > threshold:
            items.append(current_item)
            current_item = [line]
        else:
            current_item.append(line)

    items.append(current_item)
    return items


def item_to_text(item):
    return " ".join(" ".join(b["text"] for b in line) for line in item)

def create_lines(structured):
    lines = []
    for obj in structured:
        # skip adding the text if it resembles a price
        if PRICE_PATTERN.match(obj["text"]):
            continue

        placed = False
        for line in lines:
            if same_line(obj["rect"], line[-1]["rect"]):
                line.append(obj)
                placed = True
                break
        if not placed:
            lines.append([obj])
    return lines

def clean_lines(lines):
    lines_text = []

    for line in lines:
        text = ""
        for part in line:
            text = text + part["text"] + " "
        lines_text.append(text)

    return lines_text


def run_ocr(result):
    
    rect_data = structure_data(result)
    lines = create_lines(rect_data)
    #clean = clean_lines(lines)

    items = group_lines_into_items(lines)

    #items = group_menu_items(rect_data)

    #for item in items:
    #    print(item["title"])
    #    print("  ", item["description"])
    #    print()
    output = [item_to_text(item) for item in items]

    return output


import re

def tokenize_alpha(text):
    # keep only letters and spaces
    cleaned = re.sub(r'[^A-Za-z\s]', '', text)
    # collapse multiple spaces
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned


# code from https://pyimagesearch.com/2021/11/22/improving-ocr-results-with-basic-image-processing/

def preprocess_image(filename):
    image = cv2.imread(filename)

    #grayscale conversion, noise reduction, binary and adaptive thresholding, contrast enhancement, geometric correction and image sharpening

    orig = image.copy()
    ratio = image.shape[0] / 500.0

    image = cv2.resize(image, (int(image.shape[1]/ratio), 500))

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    cv2.imshow("image", gray)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if PRICE_PATTERN.match("E15.99"):
    print("match")
else:
    print("no match")

run_ocr(result)
#vocab = ['chicken', 'burger', 'soup']

#for row in ocr:
#    row = tokenize_alpha(row)
#    word_list = row.split(" ")
#    print(word_list) 


match


['Home Made Ravioli Cashew Basil Pesto, Spinach; Basil, Garlic, Parmesan; Aubergine Tomato Parmigiano, Pine Nuts, Olive Oil (G/D/N)',
 'Salmon Grilled Salmon, Green Peas Served With Mixed Salad (D) Mash E3',
 'Tuna Salad Green Salad, Green Beans, Boiled Eggs, Tuna, Cherry Tomatoes, Crystallised Walnut, French Dressing: (N/E)',
 'Gnocchi Confit Cooked Tomato With Chicken and Spinach Available GF/V + E1.99',
 'Creamy Lasagne Minced Beef; Bolognese, Parmesan (G/D)',
 'Caesar Salad Grilled Chicken, Boiled Egg, Roman Lettuce, Parmesan, Dressing, hubs and Salt Crisps ,Tomato (E/G/D)',
 'Wagyu Beef Burger Served With Potato Wedges, Brioche Bun, Tomato, Lettuce, English Cheddar (E/G/D)',
 'Chicken Tikka Wrap Salad Or Potato Wedges, Lettuce, Onion, Tomato, Chilli Sauce, Caesar Dressing (E/G/D)',
 'Falafel Wrap Falafel Salad Served With Salad Or Potato Wedges, Lettuce, Onion, Tomato, Hummus; Chilli Sauce (G)',
 'Paris Cafe and Restaurant, 63 Park Road, London NWI 6XU pariscafe63eoutlook.com']

In [28]:
from ocr_engine import run_ocr as import_run_ocr

import_run_ocr('samplemenu.jpg')

c:\Users\enara\anaconda3\envs\ocr\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


['MENU ',
 'APPETIZER ',
 'PRAWN & AVOCADO SALAD ',
 'LOCAL ROCK OYSTERS WITH VINAIGRETTE ',
 'PUMPKIN, SAGE & PARMESAN RAVIOLI ',
 'ENTREE ',
 'BEEF CARPACCIO WITH CAPERS IN OLIVE OIL ',
 'RAW KINGFISH TARTARE & EDAMAME PUREE ',
 'PRosCiutto MICRO HERB SALAD ',
 'MAIN ',
 'AGED EYE FILLET STEAK WITH FRIES ',
 'CRISPY SKIN SALMON FILLET WITH GREEN BEANS ',
 'SPAGHETTI BOLOGNESE WITH PARMESAN ',
 'DES SERT ',
 'CITRUS & THYME TART WITH DOUBLE CREAM ',
 'CARAMEL APPLE PIE WITH VANILLA BEAN ICE CREAM ',
 'ICE CREAM CHOCOLATE, VANILLA & STRAWBERRY ']